# 2. The validator, and what it has to say when it refuses

**What you get out of this notebook:** the three layers of Algorithm 2, each one
refusing a different bad reply, and the reason it produces travelling into the
repair prompt.

A validator that answers only *invalid* makes bounded repair useless: the model
gets told that something is wrong and not what. Each layer here names the fault.

In [1]:
import _bootstrap

import tsp_transient as tsp

## Layer 1 — the envelope and the schema

Is there an answer in this text at all, and does it claim the representation we asked for?

In [2]:
try:
    tsp.parse('Sure! A good tour would be 0 -> 1 -> 2 -> 3 -> 4.')
except ValueError as err:
    print(err)

schema error: no CANDIDATE envelope


## Layer 2 — the syntax of the payload

The envelope is there, but what it carries is not a list of integers.

In [3]:
try:
    tsp.parse('CANDIDATE\nid: t\nrepresentation: permutation\npayload:\nthe usual order\nEND_CANDIDATE')
except ValueError as err:
    print(err)

syntax error: no payload list


## Layer 3 — feasibility in the problem

This one parses: it is a list of integers. It is still not a tour. Note that the
message names the cities, and that is deliberate.

In [4]:
bad = tsp.parse(tsp_pool_first := __import__('llm').load_pool('tsp_pool.txt')[0])
print('parsed payload:', bad)
try:
    tsp.feasible(bad)
except ValueError as err:
    print(err)

parsed payload: [0, 1, 4, 4, 2]
feasibility: city [4] duplicated, city [3] missing


## Why the wording of the diagnostic is part of the design

The reason goes straight into the next prompt. This is the whole of bounded
repair: same prompt, plus the fault, one more attempt.

In [5]:
spec = tsp.Spec()
prompt = spec.render([0, 2, 3, 4, 1], 9.236, [])
try:
    tsp.feasible(bad)
except ValueError as err:
    print(spec.repair_prompt(prompt, '', str(err)))

[context] TSP toy instance; minimize Euclidean closed-tour length.
[conditioning] R = permutation of [0, 1, 2, 3, 4] starting at 0; incumbent [0, 2, 3, 4, 1], score 9.236.
[instruction] Emit one lower-length tour if possible.
[format] Use the CANDIDATE envelope.
[repair] previous payload failed validation: feasibility: city [4] duplicated, city [3] missing. Emit a corrected CANDIDATE.


The retry budget is a number you choose, and it is bounded on purpose: a model
that cannot fix its answer in one or two attempts will usually not fix it in ten,
and every attempt is paid for.

Next: [3. A transient operator](03_tsp_transient.ipynb).